# Appendix B: Netgen/NGSolve cheat sheet

The library [Netgen/NGSolve (NGPy)](https://ngsolve.org/) is an open-source mesher/solver for finite element problems, similar to [FEniCS](https://fenicsproject.org/), [deal.II](https://dealii.org/), [MFEM](https://mfem.org/), [OneLab](https://onelab.info/), [FreeFem++](https://freefem.org/)... 

Netgen handles geometry and mesh generation. NGSolve focuses on matrix assembly and solving the finite element problem. Interfaces are possible with other meshers/solvers such as GMSH or SciPy.

This document aims to present the library's basic functions; more details and application examples are available on the [documentation](https://docu.ngsolve.org/latest/).



____
## 1) Geometry and mesh

### 1.a) Basic usage

In [ ]:
from ngsolve import unit_square, Mesh
from ngsolve.webgui import Draw                 # to visualize mesh and fields within Jupyter

geometry = unit_square                          # simple predefined square geometry
mesh_netgen = geometry.GenerateMesh(maxh = 0.1, # to control the mesh size
                                    )
mesh_ngsolve = Mesh(mesh_netgen)                # convert netgen mesh to ngsolve format
Draw(mesh_ngsolve)                              # visualize the mesh in the notebook    

### 1.b) More elaborated 2D geometry

In [ ]:
from netgen.geom2d import SplineGeometry # one of the geometry definition modules of Netgen

geo = SplineGeometry()              # create the geometry object

# Add points
coordinates = [(0,0), (1, 0),(1, 1), (0, 1)]                        # define the coordinates of the points
points = [geo.AppendPoint(xy[0], xy[1]) for xy in coordinates]      # add the points and get their indices

# Domains
outside = 0     # exterior domain should always have index 0
triangle = 1
disk = 2

# Add lines
geo.Append(["line", points[0], points[1]],  # straight line going from points[0] to points[1]
           leftdomain=triangle,             # on the left side of the line is "triangle" domain
           rightdomain=outside,             # on the right side of the line is "disk" domain
           bc = 'bottom')                   # the line label is "bottom" ("bc" is short for "boundary condition")
geo.Append(["line", points[1], points[3]], leftdomain=triangle, rightdomain=disk, bc = 'border') 
geo.Append(["line", points[3], points[0]], leftdomain=triangle, rightdomain=outside, bc = 'left') 
geo.Append(["spline3", points[1], points[2], points[3]], # curved line defined by 3 points (start, control, end).
           leftdomain=disk, rightdomain=outside, bc = 'right')

# Add surfaces labels
geo.SetMaterial(triangle,'triangle') # Pour faciliter l'indentification des différents domaines, ils peuvent être nommé. 
geo.SetMaterial(disk,    'disk') # Pour faciliter l'indentification des différents domaines, ils peuvent être nommé. 

# Mesh generation
ngmesh = geo.GenerateMesh(maxh = 0.2)
curveOrder = 1 # to control the order of the geometry approximation (1: straight lines, 2: parabolic curves, etc.)
mesh = Mesh(ngmesh).Curve(curveOrder)
print(f"Number of vertices: {mesh.nv}")
print(f"Number of elements: {mesh.ne}")
Draw(mesh)

____
## 2) Finite element functions

### 2.a) Function spaces
We use only 2 types of functions spaces:
- $H^1$ discretized by node-based continuous functions
- $L^2$ discretized by element-based discontinuous functions

Some degrees may actually be frozen, as with **Dirichlet** boundary conditions. Just give the labels; regular expression are supported.

In [ ]:
from ngsolve import H1, L2

space1 = H1(mesh, dirichlet = "left|right")  # dofs belonging to "left" and "right" boundaries are frozen
space2 = L2(mesh)

### 2.b) Grid functions

Grid-functions can be defined on finite element spaces, attached to a specific grid (mesh).

In [ ]:
from ngsolve import GridFunction, BND

# on H1 space...
gfu = GridFunction(space1)           # create a scalar grid-function defined on "space1"
gfu.Set(1, BND)                      # set the values of Dirichlet boundary to 1
Draw(gfu)

### 2.c) Coefficient functions

Fields can also be defined independantly from any mesh.

In [ ]:
from ngsolve import CoefficientFunction, x, y

field = CoefficientFunction((x**2, y**2))  # create a vector field
Draw(field, mesh, # a coefficient function can be drawn if we provide the mesh on which it has to be evaluated
     vectors = { "grid_size":20}) # we can also draw the vectors. 

## 3) Weak formulation

As an example, we want to find $u\in H = \{H^1(\Omega), v|_{\Gamma_D}=0 \}$ consider the weak formulation :

$$ \forall v\in \int_\Omega \text{grad}(v) \cdot \nu \text{grad}(u) = \int_{\Omega_2} v j $$

### 3.a) Symbolic expressions

In [ ]:
fes = H1(mesh, dirichlet = "left|right")    # define the finite element space
u = fes.TrialFunction()                     # define the trial function (unknown solution)
v = fes.TestFunction()                      # define the test function

from ngsolve import BilinearForm, LinearForm, dx, grad

bf = BilinearForm(fes)                      # create the bilinear form object, containing u and v
nu = 1
bf += grad(v) * nu * grad(u) * dx           # write the symbolic expression, don't forget the dx

lf = LinearForm(fes)                        # create the linear form object, containing v but not u
j = 10
lf += v * j * dx("disk")                    # we can restrict the integration to a specific domain (here "disk")

### 3.b) Assemble and solve the linear system

The assembly process compute the discretization of the symbolic expressions, i.e. a matrix and right-hand side vector that can be used to solve the problem.

In [ ]:
K = bf.Assemble().mat       # assemble the finite element matrix
f = lf.Assemble().vec       # assemble the right hand side

Kinv = K.Inverse(freedofs = fes.FreeDofs(),     # indicate the free dofs (i.e. the ones that are not fixed by Dirichlet boundary conditions)
                 inverse = "sparsecholesky")    # specify the solver type, here a sparse Cholesky direct solver.

sol = GridFunction(fes)     # initialize the solution
sol.vec.data = Kinv * f     # compute the solution
Draw(sol)                   # Draw te solution

## 4) Post-processing

### 4.1) Auxiliary quantities

In [ ]:
from ngsolve import CF #shortcut for CoefficientFunction

R = CF(((0,1),(-1,0)), dims = (2,2))  # create the operator [0,1; -1,0]
b = R*grad(sol)
Draw(b, mesh, vectors = { "grid_size":20},
     settings = {"Objects" : {"Surface" : False}}) # we can tune the other settings

### 4.2) Integrals

Integrated quantities are often useful and can be computed easily with NGSolve.

In [ ]:
from ngsolve import Integrate

energy = Integrate(nu/2 * grad(sol)**2, mesh)
print(f"{energy = :.3e} J/m")

To restrict the integral to a subdomain, several syntaxes are available.

In [ ]:

energy_triangle = Integrate(nu/2 * grad(sol)**2 * dx("triangle"), mesh)
energy_disk = Integrate(nu/2 * grad(sol)**2, mesh, definedon = mesh.Materials("disk"))

print(f"{energy_triangle = :.3e} J/m")
print(f"{energy_disk = :.3e} J/m")